In [1]:
!pip install lxml tqdm

In [3]:
import os
import gzip
from lxml import etree
from tqdm import tqdm
import pandas as pd

In [4]:
def parse_medline_xml(file_path, max_articles=None):

    records = []
    article_count = 0

    # Handle compressed and uncompressed XML
    if file_path.endswith(".gz"):
        file_obj = gzip.open(file_path, "rb")
    else:
        file_obj = open(file_path, "rb")

    context = etree.iterparse(file_obj, events=("end",), tag="PubmedArticle")

    for event, elem in tqdm(context):

        try:
            # PMID
            pmid = elem.findtext(".//PMID")

            # Title
            title = elem.findtext(".//ArticleTitle")

            # Abstract (can have multiple sections)
            abstract_texts = elem.findall(".//AbstractText")
            abstract = " ".join(
                [a.text for a in abstract_texts if a is not None and a.text]
            )

            # Year
            year = elem.findtext(".//PubDate/Year")

            # Journal
            journal = elem.findtext(".//Journal/Title")

            # Publication types
            publication_types = [
                pt.text for pt in elem.findall(".//PublicationType")
                if pt.text
            ]

            # Extract MeSH terms (IMPORTANT FIX)
            mesh_terms = []

            for mh in elem.findall(".//MeshHeading"):
                descriptor = mh.find("DescriptorName")

                if descriptor is not None:
                    mesh_terms.append({
                        "name": descriptor.text.lower().strip(),
                        "ui": descriptor.get("UI")
                    })

            if abstract:  # keep only articles with abstracts
                records.append({
                    "pmid": pmid,
                    "title": title,
                    "abstract": abstract,
                    "year": year,
                    "journal": journal,
                    "publication_types": publication_types,
                    "mesh_terms": mesh_terms
                })

            article_count += 1

            if max_articles and article_count >= max_articles:
                break

        except Exception:
            pass

        # Clear memory
        elem.clear()

    file_obj.close()

    return pd.DataFrame(records)

In [5]:
base_paths = [
    "/kaggle/input/datasets/pakyalakshmi/pubmed26march"
]

xml_files = []

for base in base_paths:
    for f in os.listdir(base):
        if f.endswith(".xml") or f.endswith(".xml.gz"):
            xml_files.append(os.path.join(base, f))

print("Files found:", len(xml_files))
for f in xml_files:
    print(f)

Files found: 12
/kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0008.xml
/kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0003.xml
/kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0010.xml
/kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0070.xml
/kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0005.xml
/kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0001.xml
/kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0071.xml
/kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0069.xml
/kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0072.xml
/kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0006.xml
/kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0007.xml
/kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0002.xml


In [6]:
all_dfs = []

for file in xml_files:

    print("\nProcessing:", file)

    df = parse_medline_xml(file)

    print("Articles parsed:", len(df))

    all_dfs.append(df)

medline_df = pd.concat(all_dfs, ignore_index=True)

print("\nTotal articles:", len(medline_df))


Processing: /kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0008.xml


30000it [00:06, 4304.45it/s]


Articles parsed: 15397

Processing: /kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0003.xml


30000it [00:06, 4703.33it/s]


Articles parsed: 12659

Processing: /kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0010.xml


...

Articles parsed: 16385

Processing: /kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0007.xml


30000it [00:08, 3711.82it/s]


Articles parsed: 16407

Processing: /kaggle/input/datasets/pakyalakshmi/pubmed26march/pubmed26n0002.xml


30000it [00:07, 4228.51it/s]


Articles parsed: 13455

Total articles: 203428


In [7]:
medline_df.head()

,pmid,title,abstract,year,journal,publication_types,mesh_terms
0,214403,Carcinogenesis of N-nitrosodiethylamine (DENA)...,The environmental carcinogen N-nitrosodiethyla...,1978,International journal of cancer,[Journal Article],"[{'name': 'adenocarcinoma', 'ui': 'D000230'}, ..."
1,214404,Circulating immune complexes in rats bearing c...,Sera from rats bearing intraperitoneal implant...,1978,International journal of cancer,[Journal Article],"[{'name': 'animals', 'ui': 'D000818'}, {'name'..."
2,214405,Long-term T-cell-mediated immunity to Epstein-...,Peripheral blood mononuclear cells from donors...,1978,International journal of cancer,[Journal Article],"[{'name': 'adult', 'ui': 'D000328'}, {'name': ..."
3,214407,Antibodies to Herpes simplex virus types 1 and...,Antibody activity to Herpes simplex virus type...,1978,International journal of cancer,[Journal Article],"[{'name': 'adult', 'ui': 'D000328'}, {'name': ..."
4,214406,Malignant lymphoma in four of five siblings.,In one family four out of five siblings were a...,1978,International journal of cancer,"[Case Reports, Journal Article]","[{'name': 'antibodies, viral', 'ui': 'D000914'..."


In [8]:
medline_df.iloc[0]["mesh_terms"]

[{'name': 'adenocarcinoma', 'ui': 'D000230'},
 {'name': 'administration, oral', 'ui': 'D000284'},
 {'name': 'animals', 'ui': 'D000818'},
 {'name': 'animals, domestic', 'ui': 'D000829'},
 {'name': 'carcinogens', 'ui': 'D002273'},
 {'name': 'carcinoma, hepatocellular', 'ui': 'D006528'},
 {'name': 'cats', 'ui': 'D002415'},
 {'name': 'chickens', 'ui': 'D002645'},
 {'name': 'diethylnitrosamine', 'ui': 'D004052'},
 {'name': 'female', 'ui': 'D005260'},
 {'name': 'injections, intramuscular', 'ui': 'D007273'},
 {'name': 'kidney neoplasms', 'ui': 'D007680'},
 {'name': 'liver neoplasms', 'ui': 'D008113'},
 {'name': 'male', 'ui': 'D008297'},
 {'name': 'neoplasms, experimental', 'ui': 'D009374'},
 {'name': 'nitrosamines', 'ui': 'D009602'},
 {'name': 'species specificity', 'ui': 'D013045'}]

In [8]:
medline_df.to_csv("parsed_medline_dataset.csv", index=False)

In [9]:
from lxml import etree
from tqdm import tqdm
import pandas as pd

In [10]:
mesh_descriptor_path = "/kaggle/input/datasets/pakyalakshmi/mesh-descriptor/desc2026.xml"

mesh_records = []

context = etree.iterparse(mesh_descriptor_path, events=("end",), tag="DescriptorRecord")

for event, elem in tqdm(context):

    try:
        ui = elem.findtext("DescriptorUI")

        name = elem.findtext(".//DescriptorName/String")

        tree_numbers = [
            tn.text for tn in elem.findall(".//TreeNumber")
        ]

        for tree in tree_numbers:

            mesh_records.append({
                "mesh_ui": ui,
                "mesh_name": name.lower().strip(),
                "tree_number": tree
            })

    except Exception:
        pass

    elem.clear()

mesh_descriptor_df = pd.DataFrame(mesh_records)

print("Total records:", len(mesh_descriptor_df))
mesh_descriptor_df.head()

31110it [00:07, 4038.15it/s]

Total records: 65360


,mesh_ui,mesh_name,tree_number
0,D000001,calcimycin,D02.355.291.933.125
1,D000001,calcimycin,D02.540.576.625.125
2,D000001,calcimycin,D03.633.100.221.173
3,D000001,calcimycin,D04.345.241.654.125
4,D000001,calcimycin,D04.345.674.625.125


In [11]:
mesh_descriptor_df["mesh_ui"].nunique()

31108

In [12]:
def classify_mesh_tree(tree_number):

    prefix = tree_number[0]

    if prefix == "C":
        return "Disease"
    elif prefix == "D":
        return "Chemical"
    elif prefix == "A":
        return "Anatomy"
    elif prefix == "G":
        return "Biological_Process"
    elif prefix == "E":
        return "Technique"
    else:
        return "Other"

In [13]:
mesh_descriptor_df["node_type"] = mesh_descriptor_df["tree_number"].apply(classify_mesh_tree)

mesh_descriptor_df.head()

,mesh_ui,mesh_name,tree_number,node_type
0,D000001,calcimycin,D02.355.291.933.125,Chemical
1,D000001,calcimycin,D02.540.576.625.125,Chemical
2,D000001,calcimycin,D03.633.100.221.173,Chemical
3,D000001,calcimycin,D04.345.241.654.125,Chemical
4,D000001,calcimycin,D04.345.674.625.125,Chemical


In [14]:
mesh_descriptor_df["node_type"] = mesh_descriptor_df["tree_number"].apply(classify_mesh_tree)

mesh_ui_lookup = (
    mesh_descriptor_df
    .sort_values("tree_number")
    .drop_duplicates(subset="mesh_ui")
)

mesh_ui_lookup = mesh_ui_lookup[["mesh_ui", "mesh_name", "node_type"]]

print("Unique MeSH descriptors:", len(mesh_ui_lookup))
mesh_ui_lookup.head()

Unique MeSH descriptors: 31108


,mesh_ui,mesh_name,node_type
3355,D001829,body regions,Anatomy
54474,D059925,anatomic landmarks,Anatomy
3577,D001940,breast,Anatomy
44134,D042361,"mammary glands, human",Anatomy
18236,D009558,nipples,Anatomy


In [15]:
mesh_ui_lookup.to_csv("mesh_ui_lookup.csv", index=False)

In [16]:
import pandas as pd
import ast
from tqdm import tqdm

mesh_lookup = pd.read_csv("mesh_ui_lookup.csv")

print("MeSH descriptors:", len(mesh_lookup))

MeSH descriptors: 31108


In [17]:
cardio_mesh = mesh_descriptor_df[
    mesh_descriptor_df["tree_number"].str.startswith("C14")
]["mesh_ui"].unique()

len(cardio_mesh)

471

In [18]:
def is_cardio_article(mesh_terms):
    
    for mesh in mesh_terms:
        if mesh["ui"] in cardio_mesh:
            return True
            
    return False


cardio_df = medline_df[
    medline_df["mesh_terms"].apply(is_cardio_article)
]

print("Cardiovascular articles:", len(cardio_df))

Cardiovascular articles: 14893


In [19]:
node_records = []

for _, row in tqdm(cardio_df.iterrows(), total=len(cardio_df)):

    pmid = row["pmid"]
    year = row["year"]

    for mesh in row["mesh_terms"]:

        node_records.append({
            "pmid": pmid,
            "year": year,
            "mesh_ui": mesh["ui"],
            "node_name": mesh["name"]
        })

nodes_df = pd.DataFrame(node_records)

100%|██████████| 14893/14893 [00:00<00:00, 22453.37it/s]


In [20]:
nodes_df = nodes_df.merge(
    mesh_lookup,
    on="mesh_ui",
    how="left"
)

core_nodes_df = nodes_df[
    nodes_df["node_type"].isin([
        "Disease",
        "Chemical",
        "Biological_Process",
        "Anatomy"
    ])
]

In [21]:
unique_nodes = nodes_df.drop_duplicates(subset=["mesh_ui"])

unique_nodes = unique_nodes[
    ["mesh_ui", "node_name", "node_type"]
]

print("Unique nodes:", len(unique_nodes))
unique_nodes.head()

Unique nodes: 7858


,mesh_ui,node_name,node_type
0,D003937,"diagnosis, differential",Technique
1,D005918,glomus tumor,Disease
2,D006390,hemangioendothelioma,Disease
3,D006391,hemangioma,Disease
4,D006392,"hemangioma, cavernous",Disease


In [22]:
unique_nodes = unique_nodes.reset_index(drop=True)

unique_nodes["node_id"] = [
    f"N{i+1}" for i in range(len(unique_nodes))
]

unique_nodes = unique_nodes[
    ["node_id", "node_name", "mesh_ui", "node_type"]
]

unique_nodes.head()

,node_id,node_name,mesh_ui,node_type
0,N1,"diagnosis, differential",D003937,Technique
1,N2,glomus tumor,D005918,Disease
2,N3,hemangioendothelioma,D006390,Disease
3,N4,hemangioma,D006391,Disease
4,N5,"hemangioma, cavernous",D006392,Disease


In [139]:
nodes_df = unique_nodes

In [140]:
entity_names = set(nodes_df["node_name"].str.lower())
print("Total entities:", len(entity_names))

Total entities: 7858


In [141]:
import nltk
from tqdm import tqdm

nltk.download("punkt")

sentence_records = []

for _, row in tqdm(cardio_df.iterrows(), total=len(cardio_df)):

    pmid = row["pmid"]
    abstract = str(row["abstract"])

    sentences = nltk.sent_tokenize(abstract)

    for s in sentences:
        sentence_records.append({
            "pmid": pmid,
            "sentence": s.lower()
        })

sent_df = pd.DataFrame(sentence_records)

print("Total sentences:", len(sent_df))

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
100%|██████████| 14893/14893 [00:02<00:00, 5381.01it/s]


Total sentences: 101107


In [27]:
def find_entities(sentence):

    found = []

    for entity in entity_names:
        if entity in sentence:
            found.append(entity)

    return list(set(found))


sent_df["entities"] = sent_df["sentence"].apply(find_entities)

In [28]:
candidate_sentences = sent_df[
    sent_df["entities"].apply(lambda x: len(x) >= 2)
]

print("Candidate sentences:", len(candidate_sentences))

Candidate sentences: 67833


In [29]:
relation_patterns = {
    "increases_risk_of": [
        "increases risk of",
        "risk factor for",
        "associated with increased risk of"
    ],

    "causes": [
        "causes",
        "leads to",
        "results in"
    ],

    "treated_by": [
        "treated with",
        "treated by",
        "therapy with"
    ],

    "associated_with": [
        "associated with",
        "correlated with",
        "linked to"
    ]
}

In [30]:
edges = []

for _, row in tqdm(candidate_sentences.iterrows(), total=len(candidate_sentences)):

    sentence = row["sentence"]
    entities = row["entities"]

    for relation, patterns in relation_patterns.items():

        for pattern in patterns:

            if pattern in sentence:

                for e1 in entities:
                    for e2 in entities:

                        if e1 != e2:

                            edges.append({
                                "source": e1,
                                "target": e2,
                                "relation": relation,
                                "sentence": sentence
                            })

100%|██████████| 67833/67833 [00:02<00:00, 24805.11it/s]


In [31]:
edges_df = pd.DataFrame(edges).drop_duplicates()

print("Edges extracted:", len(edges_df))
edges_df.head()

Edges extracted: 72720


,source,target,relation,sentence
0,animals,methylprednisolone,treated_by,hearts were obtained from both control animals...
1,animals,prednisolone,treated_by,hearts were obtained from both control animals...
2,animals,heart,treated_by,hearts were obtained from both control animals...
3,animals,ear,treated_by,hearts were obtained from both control animals...
4,methylprednisolone,animals,treated_by,hearts were obtained from both control animals...


In [37]:
nodes_df.columns

Index(['node_id', 'node_name', 'mesh_ui', 'node_type'], dtype='object')

In [32]:
node_map = dict(zip(nodes_df["node_name"], nodes_df["node_id"]))

In [33]:
edges_df["source_id"] = edges_df["source"].map(node_map)
edges_df["target_id"] = edges_df["target"].map(node_map)

In [34]:
edges_df = edges_df.dropna(subset=["source_id", "target_id"])

In [35]:
edges_weighted = (
    edges_df
    .groupby(["source_id", "relation", "target_id"])
    .size()
    .reset_index(name="weight")
)

print("Unique edges:", len(edges_weighted))
edges_weighted.head()

Unique edges: 43382


,source_id,relation,target_id,weight
0,N10,associated_with,N1079,1
1,N10,associated_with,N1285,1
2,N10,associated_with,N1995,1
3,N10,associated_with,N1996,1
4,N10,associated_with,N3288,1


In [36]:
node_lookup = dict(zip(unique_nodes["node_id"], unique_nodes["node_name"]))

edges_weighted["source"] = edges_weighted["source_id"].map(node_lookup)
edges_weighted["target"] = edges_weighted["target_id"].map(node_lookup)

edges_weighted.head()

,source_id,relation,target_id,weight,source,target
0,N10,associated_with,N1079,1,lymphedema,edema
1,N10,associated_with,N1285,1,lymphedema,blood
2,N10,associated_with,N1995,1,lymphedema,lymph
3,N10,associated_with,N1996,1,lymphedema,lymphatic system
4,N10,associated_with,N3288,1,lymphedema,lip


In [37]:
unique_nodes.to_csv("graph_nodes.csv", index=False)

edges_weighted.to_csv("graph_edges.csv", index=False)

In [185]:
edges_clean = edges_weighted[
    edges_weighted["weight"] >= 3
]

print("Edges after filtering:", len(edges_clean))

Edges after filtering: 4250


In [186]:
connected_nodes = set(edges_clean["source_id"]).union(
    set(edges_clean["target_id"])
)

nodes_clean = unique_nodes[
    unique_nodes["node_id"].isin(connected_nodes)
]

print("Remaining nodes:", len(nodes_clean))

Remaining nodes: 394


In [187]:
nodes_clean.to_csv("cardiograph_nodes.csv", index=False)

edges_clean.to_csv("cardiograph_edges.csv", index=False)

In [87]:
!pip install networkx sentence-transformers

In [41]:
import networkx as nx
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

In [188]:
nodes_df = nodes_clean
edges_df = edges_clean

In [189]:
G = nx.DiGraph()

# Add nodes
for _, row in nodes_df.iterrows():
    
    G.add_node(
        row["node_id"],
        name=row["node_name"],
        type=row["node_type"]
    )

# Add edges
for _, row in edges_df.iterrows():
    
    G.add_edge(
        row["source_id"],
        row["target_id"],
        relation=row["relation"],
        weight=row["weight"]
    )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 394
Edges: 3106


In [190]:
node_lookup = dict(zip(nodes_df["node_name"], nodes_df["node_id"]))

def entity_linking(query):

    query = query.lower()

    matches = []

    query_tokens = query.split()

    for name, node_id in node_lookup.items():

        name = name.lower()

        for token in query_tokens:

            if token in name:
                matches.append(node_id)
                break

    return list(set(matches))

In [191]:
entity_linking("Does hypertension cause stroke?")

['N3988', 'N49']

In [192]:
for nid in entity_linking("Does hypertension cause stroke?"):
    print(nid, G.nodes[nid]["name"])

N3988 cause of death
N49 hypertension


In [47]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = sent_df["sentence"].tolist()

sentence_embeddings = model.encode(
    sentences,
    show_progress_bar=True
)

sentence_embeddings = np.array(sentence_embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3160 [00:00<?, ?it/s]

In [48]:
sentence_embeddings.shape

(101107, 384)

In [50]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 77.2 MB/s eta 0:00:00:00:0100:01


In [51]:
import faiss
import numpy as np

sentence_embeddings = sentence_embeddings.astype("float32")

dimension = sentence_embeddings.shape[1]

faiss_index = faiss.IndexFlatL2(dimension)

faiss_index.add(sentence_embeddings)

print("FAISS index size:", faiss_index.ntotal)

FAISS index size: 101107


In [52]:
import numpy as np
import faiss

sentence_embeddings = sentence_embeddings.astype("float32")

faiss.normalize_L2(sentence_embeddings)

In [53]:
dimension = sentence_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(dimension)

faiss_index.add(sentence_embeddings)

print("FAISS index size:", faiss_index.ntotal)

FAISS index size: 101107


In [54]:
def faiss_retrieve(query, k=20):

    query_vec = model.encode([query]).astype("float32")

    faiss.normalize_L2(query_vec)

    scores, indices = faiss_index.search(query_vec, k)

    retrieved = sent_df.iloc[indices[0]].copy()

    retrieved["score"] = scores[0]

    return retrieved

In [55]:
faiss_results = faiss_retrieve(
    "Does hypertension increase stroke risk?",
    k=50
)

faiss_results.head()

,pmid,sentence,entities,score
14012,2095387,the increasing tendency to treat hypertension ...,"[disease, mortality, hypertension, heart, ear]",0.743904
69431,2082449,"hypertension, diabetes mellitus, smoking and h...","[disease, risk, smoking, diabetes mellitus, ri...",0.742447
60830,2063853,treatment of hypertension has decreased the in...,"[incidence, hypertension, heart, ear, heart fa...",0.730867
59227,2150640,hypertension is a major risk factor for corona...,"[disease, risk, hypertension, heart, ear]",0.727470
15158,2097347,"however, the rise observed in hypertensive str...","[hypertension, tin]",0.718216


In [147]:
entity_names = set(nodes_clean["node_name"].str.lower())

In [148]:
def detect_entities(sentence):

    sentence = sentence.lower()

    found = []

    for e in entity_names:
        if e in sentence:
            found.append(e)

    return list(set(found))

In [149]:
faiss_results["entities"] = faiss_results["sentence"].apply(detect_entities)

faiss_results = faiss_results[
    faiss_results["entities"].apply(len) > 0
]

In [59]:
node_map = dict(
    zip(nodes_clean["node_name"], nodes_clean["node_id"])
)

def map_entities(entity_list):

    return [
        node_map[e] for e in entity_list
        if e in node_map
    ]

faiss_results["node_ids"] = faiss_results["entities"].apply(map_entities)

In [60]:
def multi_hop_paths(start_nodes, max_hops=2):

    paths = []

    for node in start_nodes:

        for target in nx.single_source_shortest_path(
            G,
            node,
            cutoff=max_hops
        ):

            if target != node:

                path = nx.shortest_path(G, node, target)

                paths.append(path)

    return paths

In [61]:
start_nodes = set()

for nodes in faiss_results["node_ids"]:
    start_nodes.update(nodes)

start_nodes = list(start_nodes)

In [62]:
paths = multi_hop_paths(start_nodes, max_hops=2)

print("Candidate paths:", len(paths))

Candidate paths: 14287


In [193]:
node_names = nodes_df["node_name"].tolist()

node_embeddings = model.encode(
    node_names,
    show_progress_bar=True
)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

In [194]:
import faiss
import numpy as np

node_embeddings = np.array(node_embeddings).astype("float32")

faiss.normalize_L2(node_embeddings)

In [195]:
dimension = node_embeddings.shape[1]

node_index = faiss.IndexFlatIP(dimension)

node_index.add(node_embeddings)

print("Node index size:", node_index.ntotal)

Node index size: 394


In [196]:
def entity_linking(query, k=5):

    query_vec = model.encode([query]).astype("float32")

    faiss.normalize_L2(query_vec)

    scores, indices = node_index.search(query_vec, k)

    nodes = nodes_df.iloc[indices[0]]

    return nodes[["node_id", "node_name"]]

In [197]:
entity_linking("Does hypertension cause stroke?")

,node_id,node_name
48,N49,hypertension
1005,N1006,cerebral arteries
458,N459,cardiovascular diseases
623,N624,cerebral infarction
322,N323,vascular diseases


In [198]:
def string_entity_linking(query):

    query = query.lower()

    matches = []

    for name, node_id in node_lookup.items():

        if name in query:
            matches.append(node_id)

    return matches

In [199]:
def embedding_entity_linking(query, k=5):

    query_vec = model.encode([query]).astype("float32")

    faiss.normalize_L2(query_vec)

    scores, indices = node_index.search(query_vec, k)

    nodes = nodes_df.iloc[indices[0]]

    return nodes["node_id"].tolist()

In [200]:
def hybrid_entity_linking(query, k=5):

    string_nodes = string_entity_linking(query)

    embedding_nodes = embedding_entity_linking(query, k)

    combined = list(set(string_nodes + embedding_nodes))

    return combined

In [201]:
nodes = hybrid_entity_linking(
    "Does hypertension cause stroke?"
)

for nid in nodes:

    print(nid, G.nodes[nid]["name"])

N1006 cerebral arteries
N624 cerebral infarction
N459 cardiovascular diseases
N323 vascular diseases
N49 hypertension


In [202]:
start_nodes = hybrid_entity_linking(
    "Does hypertension cause stroke?"
)

In [203]:
import networkx as nx

'''
def multi_hop_traversal(start_nodes, max_hops=2):

    paths = []

    for start in start_nodes:

        if start not in G:
            continue

        for target in nx.single_source_shortest_path(
            G,
            start,
            cutoff=max_hops
        ):

            if target == start:
                continue

            path = nx.shortest_path(G, start, target)

            paths.append(path)

    return paths
'''
def multi_hop_traversal(start_nodes, max_hops=2, min_weight=3):

    paths = []

    for start in start_nodes:

        if start not in G:
            continue

        queue = [(start, [start])]

        while queue:

            node, path = queue.pop(0)

            if len(path) > max_hops + 1:
                continue

            for neighbor in G.successors(node):

                edge_weight = G[node][neighbor]["weight"]

                # Only follow strong edges
                if edge_weight < min_weight:
                    continue

                new_path = path + [neighbor]

                paths.append(new_path)

                queue.append((neighbor, new_path))

    return paths

In [231]:
generic_words = {
    "blood","pressure","risk","patients","patient",
    "group","study","level","levels","rate","rates",
    "increase","decrease","effect","effects", "lead"
}

nodes_clean = nodes_clean[
    ~nodes_clean["node_name"].str.lower().isin(generic_words)
]

In [232]:
nodes_clean = nodes_clean[
    nodes_clean["node_type"].isin([
        "Disease",
        "Chemical",
        "Biological_Process",
        "Anatomy"
    ])
]

In [233]:
G = nx.DiGraph()

for _, row in nodes_clean.iterrows():

    G.add_node(
        row["node_id"],
        name=row["node_name"],
        type=row["node_type"]
    )

for _, row in edges_clean.iterrows():

    if row["source_id"] in nodes_clean["node_id"].values and \
       row["target_id"] in nodes_clean["node_id"].values:

        G.add_edge(
            row["source_id"],
            row["target_id"],
            relation=row["relation"],
            weight=row["weight"]
        )

In [207]:
def hybrid_entity_linking(query, k=5):

    nodes = embedding_entity_linking(query, k)

    disease_nodes = nodes_df[
        nodes_df["node_id"].isin(nodes) &
        (nodes_df["node_type"] == "Disease")
    ]

    return disease_nodes["node_id"].tolist()

In [208]:
nodes_clean.sample(20)

,node_id,node_name,mesh_ui,node_type
4129,N4130,sarcoplasmic reticulum,D012519,Anatomy
5109,N5110,lead,D007854,Chemical
73,N74,collagen,D003094,Chemical
2110,N2111,cells,D002477,Anatomy
80,N81,chlorothiazide,D002740,Chemical
1301,N1302,thromboembolism,D013923,Disease
1078,N1079,edema,D004487,Disease
1029,N1030,nephrotic syndrome,D009404,Disease
6381,N6382,ticlopidine,D013988,Chemical
1137,N1138,azathioprine,D001379,Chemical


In [165]:
node_freq = edges_clean["source_id"].value_counts() + edges_clean["target_id"].value_counts()

In [85]:
valid_nodes = node_freq[node_freq >= 3].index

nodes_clean = nodes_clean[
    nodes_clean["node_id"].isin(valid_nodes)
]

In [86]:
degree = dict(G.degree())

hub_nodes = [
    node for node, deg in degree.items()
    if deg > 50
]

In [87]:
nodes_clean = nodes_clean[
    ~nodes_clean["node_id"].isin(hub_nodes)
]

In [88]:
nodes_clean["node_type"].value_counts()

node_type
Chemical              81
Disease               69
Anatomy               33
Biological_Process    19
Name: count, dtype: int64

In [89]:
G = nx.DiGraph()

for _, row in nodes_clean.iterrows():

    G.add_node(
        row["node_id"],
        name=row["node_name"],
        type=row["node_type"]
    )

for _, row in edges_clean.iterrows():

    if row["source_id"] in G.nodes and row["target_id"] in G.nodes:

        G.add_edge(
            row["source_id"],
            row["target_id"],
            relation=row["relation"],
            weight=row["weight"]
        )

In [234]:
node_type_map = dict(
    zip(nodes_df["node_id"], nodes_df["node_type"])
)

In [235]:
def filtered_entity_linking(query, k=5):

    nodes = embedding_entity_linking(query, k)

    allowed_types = {"Disease", "Chemical"}

    filtered = []

    for n in nodes:

        if n in G.nodes and node_type_map.get(n) in allowed_types:
            filtered.append(n)

    return filtered

In [236]:
start_nodes = filtered_entity_linking(
    "Does hypertension cause stroke?"
)

In [237]:
nodes = filtered_entity_linking(
    "Does hypertension cause stroke?"
)

for n in nodes:
    print(n, G.nodes[n]["name"], node_type_map[n])

N49 hypertension Disease
N459 cardiovascular diseases Disease
N624 cerebral infarction Disease
N323 vascular diseases Disease


In [213]:
nodes_df[nodes_df["node_name"].str.contains("hypertension", case=False)]

,node_id,node_name,mesh_ui,node_type
48,N49,hypertension,D006973,Disease


In [238]:
import networkx as nx

G = nx.DiGraph()

for _, row in nodes_clean.iterrows():

    G.add_node(
        row["node_id"],
        name=row["node_name"],
        type=row["node_type"]
    )

for _, row in edges_clean.iterrows():

    if row["source_id"] in G.nodes and row["target_id"] in G.nodes:

        G.add_edge(
            row["source_id"],
            row["target_id"],
            relation=row["relation"],
            weight=row["weight"]
        )

In [215]:
for n in G.nodes:
    if "hypertension" in G.nodes[n]["name"].lower():
        print(n, G.nodes[n]["name"])

N49 hypertension


In [216]:
nodes = filtered_entity_linking(
    "Does hypertension cause stroke?"
)

for n in nodes:
    print(n, G.nodes[n]["name"])

N49 hypertension
N459 cardiovascular diseases
N624 cerebral infarction
N323 vascular diseases


In [239]:
# Precompute node embeddings once
nodes_df = nodes_df.reset_index(drop=True)
node_names = nodes_df["node_name"].tolist()

node_embeddings = model.encode(
    node_names,
    convert_to_numpy=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

In [240]:
node_emb_map = {
    row["node_id"]: node_embeddings[i]
    for i, row in nodes_df.iterrows()
}

In [241]:
print(len(node_emb_map))
print(node_embeddings.shape)

394
(394, 384)


In [248]:
import numpy as np

GENERIC_WORDS = {
    "risk","patients","patient","group","increase","effect",
    "study","level","rate","pressure","blood", "lead"
}

def hybrid_score(query, path):

    # ---------- semantic similarity ----------
    q_emb = model.encode(query, convert_to_numpy=True)

    path_names = [G.nodes[n]["name"] for n in path]

    # average embedding of nodes
    node_vecs = [node_emb_map[n] for n in path]
    path_vec = np.mean(node_vecs, axis=0)

    cosine = np.dot(q_emb, path_vec) / (
        np.linalg.norm(q_emb) * np.linalg.norm(path_vec)
    )

    semantic_score = (cosine + 1) / 2


    # ---------- edge evidence ----------
    weights = []

    for i in range(len(path)-1):

        if G.has_edge(path[i], path[i+1]):

            w = G[path[i]][path[i+1]]["weight"]
            weights.append(w)

    if weights:
        edge_score = np.log(np.mean(weights) + 1) / np.log(100)
    else:
        edge_score = 0


    # ---------- hop penalty ----------
    hop_score = 1 / len(path)


    # ---------- generic node penalty ----------
    penalty = 0

    for name in path_names:
        if name.lower() in GENERIC_WORDS:
            penalty += 0.2


    # ---------- final score ----------
    score = (
        0.5 * semantic_score +
        0.35 * edge_score +
        0.15 * hop_score
    )

    score -= penalty

    return score

In [249]:
def rank_paths(query, paths):

    scored_paths = []

    for p in paths:

        s = hybrid_score(query, p)

        scored_paths.append((p, s))

    scored_paths = sorted(
        scored_paths,
        key=lambda x: x[1],
        reverse=True
    )

    return scored_paths

In [250]:
def graph_retrieval(query, top_k=5):

    # entity linking
    start_nodes = filtered_entity_linking(query)

    # multi-hop traversal
    paths = multi_hop_traversal(start_nodes, max_hops=2)

    # scoring
    ranked = rank_paths(query, paths)

    return ranked[:top_k]

In [251]:
results = graph_retrieval(
    "Does hypertension increase stroke risk?"
)

for path, score in results:

    names = [G.nodes[n]["name"] for n in path]

    print(names, "→", score)

['hypertension', 'arteries'] → 0.6436970233917236
['cerebral infarction', 'infarction', 'myocardial infarction'] → 0.6397706988465874
['cerebral infarction', 'infarction', 'myocardial infarction', 'infarction'] → 0.6374510235195748
['hypertension', 'arteries', 'hypertension'] → 0.6260481238365174
['hypertension', 'ear', 'heart'] → 0.6233225049760077


In [263]:
!pip install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00:00:0100:01


In [264]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [265]:
def path_to_text(path):

    names = [G.nodes[n]["name"] for n in path]

    return " → ".join(names)

In [266]:
def retrieve_evidence(query):

    # FAISS sentences
    faiss_results = faiss_retrieve(query, k=20)

    sentences = faiss_results["sentence"].tolist()

    # graph paths
    graph_results = graph_retrieval(query, top_k=5)

    return sentences, graph_results

In [267]:
def build_context(query):

    sentences, graph_results = retrieve_evidence(query)

    graph_paths = []

    for path, score in graph_results:
        graph_paths.append(path_to_text(path))

    context = ""

    context += "Evidence from biomedical literature:\n"

    for s in sentences[:5]:
        context += "- " + s + "\n"

    context += "\nKnowledge graph relationships:\n"

    for p in graph_paths:
        context += "- " + p + "\n"

    return context

In [268]:
def build_prompt(query, context):

    prompt = f"""
You are a biomedical assistant.

Use ONLY the evidence provided below.

Evidence:
{context}

Question:
{query}

Answer clearly using biomedical reasoning.
"""

    return prompt

In [269]:
def generate_answer(query):

    context = build_context(query)

    prompt = build_prompt(query, context)

    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.2
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response

In [270]:
query = "Does hypertension increase stroke risk?"

answer = generate_answer(query)

print(answer)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



You are a biomedical assistant.

Use ONLY the evidence provided below.

Evidence:
Evidence from biomedical literature:
- the increasing tendency to treat hypertension has markedly reduced stroke mortality but has not significantly reduced the mortality from coronary heart disease.
- hypertension, diabetes mellitus, smoking and heart disease (coronary heart disease, rhythm disturbances) were identified as risk factors for stroke.
- treatment of hypertension has decreased the incidence of stroke and congestive heart failure consequential to hypertension.
- hypertension is a major risk factor for coronary heart disease (chd) and is the primary risk factor for stroke.
- however, the rise observed in hypertensive stroke group (554.26 +/- 47.08 mg%) is significantly more (p. less than 0.01) than that occurring for nonhypertensive stroke group (497.82 +/- 93.12 mg%) indicating that the presence of hypertension does contribute to the rise.

Knowledge graph relationships:
- hypertension → arte

In [ ]:
test_queries = [
"What diseases are caused by hypertension?",
"How is hypertension treated?",
"What medications treat high blood pressure?",
"How does hypertension affect blood vessels?",
"What biological processes link hypertension and stroke?",
"What diseases are associated with hypertension?",
"How does hypertension lead to stroke?",
"Can high blood pressure cause a cerebrovascular accident?",
"Does hypertension cause ear infections?"
]

for q in test_queries:

    print("\n====================")
    print("Query:", q)

    answer = generate_answer(q)

    print("Answer:", answer)